In [ ]:
import sys
sys.path.append("../..")

import mytorch
import mytorch.nn.functional as F

DEVICE = "gpu:0"

In [ ]:
from cnn_model import CnnModel
from feed_forward_model import FeedForwardModel
from vit_model import create_vit

model = create_vit()
sgd = mytorch.SGD(model.parameters(), 0.001, 0)

## Training

In [ ]:
import tqdm
from mnist_dataset import MnistDatset
from mytorch import AsyncDataLoader

NUM_EPOCHS = 20
BATCH_SIZE = 500

train_dataset = MnistDatset("dataset", True)
train_loader = AsyncDataLoader(train_dataset, BATCH_SIZE, device=DEVICE)

In [ ]:
from mytorch import MultiStepLR

scheduler = MultiStepLR(sgd, [0.6 * NUM_EPOCHS, 0.8 * NUM_EPOCHS], 0.1)

for epoch in range(NUM_EPOCHS):
    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch}")

    for X, y in pbar:
        logits = model.forward(X.reshape(-1, 1, 28, 28))
        loss = F.cross_entropy(logits, F.onehot(y, 10))
        pbar.set_postfix(loss="%.5f" % loss.item())

        sgd.zero_grad()
        loss.backward()
        sgd.step()
    
    scheduler.step()

## Inference / Test

In [ ]:
test_dataset = MnistDatset("dataset", False)
test_loader = AsyncDataLoader(test_dataset, BATCH_SIZE, device=DEVICE)

In [ ]:
import numpy as np

num_correct = 0
with mytorch.no_grad():
    for X, y in tqdm.tqdm(test_loader):
        logits = model.forward(X.reshape(-1, 1, 28, 28))
        prediction = logits.argmax(dim=1)
        prediction = prediction == y
        num_correct += np.sum(prediction.data)

print(100 * num_correct / len(test_dataset))